# 03 · Build graphs and check for leakage

One unit per dataset × graph builder. Produces edge files under `DATA_ROOT/<dataset>/processed/graphs/`.

Builders:

- **native** (DGraph-Fin): the real emergency-contact edges. Writes a *clean* variant with all test nodes removed
  (for LUPI teachers) and a time-cutoff variant (only edges with timestamp ≤ cutoff).
- **relational** (Home Credit): customer → bureau record, customer → previous application. Full and clean variants.
- **shared_attribute** (Lending Club): applicants sharing zip3 or normalised employer title, with degree caps.
- **knn** (all three tabular sets): k nearest neighbours in standardised, PCA-reduced feature space, for k in {5, 10, 20}.
- **random_control**: the kNN graph with destinations shuffled. Same nodes, same degree sequence, meaningless edges.

**Clean-room rule.** Test rows are never part of any graph used to train a teacher. kNN and shared-attribute graphs are
built over train + validation rows only. A separate *query* edge file connects each test row to train/val rows
so Tier 1–3 models can score test rows (a new applicant can be looked up against historical customers); test rows
never connect to each other and their labels are never read. The assertions below fail the unit if any clean
graph touches a test node.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 03_build_graphs dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/03_build_graphs/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "taiwan"
seed = 0
graph_builder = "knn"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "03_build_graphs"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

In [ ]:
from src.data import splits
from src.data.base import Bundle

cfg = config.load_config(dataset)
models_cfg = config.quick_override(cfg, quick)
bundle = Bundle.load(paths.processed_dir(dataset, quick))
split = splits.load_split(dataset, quick)
X, y = bundle.X, bundle.y.to_numpy()
tr, va, te = split["train"], split["val"], split["test"]
allowed = np.sort(np.concatenate([tr, va]))          # rows that may take part in graph construction
X_tr, X_va, X_te = X.iloc[tr], X.iloc[va], X.iloc[te]
y_tr, y_va, y_te = y[tr], y[va], y[te]
gdir = paths.graphs_dir(dataset, quick)
n_boot = 200 if quick else cfg["bootstrap"]["n_boot"]
print(bundle.summary())
print({p: len(split[p]) for p in splits.PARTS}, "| test positive rate:", round(float(np.nanmean(y_te)), 4))

## Build

In [ ]:
from src.data.splits import masks
from src.graphs import EdgeList, assert_no_test_nodes
from src.graphs import knn as KNN
from src.graphs import random_control as RC
from src.graphs import relational as REL
from src.graphs import shared_attribute as SA
from src.graphs.base import degree_stats, label_homophily

built = {}        # name -> EdgeList (homogeneous graphs); 'clean' ones must not touch test nodes
full_graphs = set()  # names that legitimately contain test nodes (real edges or query attachments)
stats = {}
train_mask = masks(split, bundle.n)["train"]
Xnp = X.to_numpy(dtype=np.float32)
t = time.time()

if graph_builder == "native":
    e = EdgeList.load(gdir / "edges_native.npz")
    built["native"] = e; full_graphs.add("native")
    e_clean = e.drop_nodes(te); e_clean.meta.update(builder="native", variant="clean")
    e_clean.save(gdir / "edges_native_clean.npz"); built["native_clean"] = e_clean
    cutoff = cfg["split"].get("edge_time_cutoff")
    if cutoff is not None and e.edge_time is not None:
        e_t = e.filter_time(int(cutoff)); e_t.meta.update(variant=f"t{cutoff}")
        e_t.save(gdir / f"edges_native_t{cutoff}.npz"); built[f"native_t{cutoff}"] = e_t; full_graphs.add(f"native_t{cutoff}")

elif graph_builder == "knn":
    kcfg = cfg["graph"]["knn"]
    clean, query = KNN.build_knn_all(Xnp, allowed, te, kcfg["k"], kcfg["pca_dims"], seed=0)
    for k in kcfg["k"]:
        clean[k].save(gdir / f"edges_knn_k{k}.npz"); built[f"knn_k{k}"] = clean[k]
        query[k].save(gdir / f"edges_knn_k{k}_query.npz"); built[f"knn_k{k}_query"] = query[k]; full_graphs.add(f"knn_k{k}_query")

elif graph_builder == "random_control":
    kcfg = cfg["graph"]["knn"]
    for k in kcfg["k"]:
        src_clean, src_query = gdir / f"edges_knn_k{k}.npz", gdir / f"edges_knn_k{k}_query.npz"
        if not src_clean.exists():
            print("kNN graphs missing; building them first")
            clean, query = KNN.build_knn_all(Xnp, allowed, te, kcfg["k"], kcfg["pca_dims"], seed=0)
            for kk in kcfg["k"]:
                clean[kk].save(gdir / f"edges_knn_k{kk}.npz"); query[kk].save(gdir / f"edges_knn_k{kk}_query.npz")
        e = RC.rewire(EdgeList.load(src_clean), seed=0); e.meta["variant"] = "clean"
        e.save(gdir / f"edges_random_control_k{k}.npz"); built[f"random_control_k{k}"] = e
        q = RC.rewire(EdgeList.load(src_query), seed=1); q.meta["variant"] = "query"   # test rows -> random allowed rows
        q.save(gdir / f"edges_random_control_k{k}_query.npz"); built[f"random_control_k{k}_query"] = q; full_graphs.add(f"random_control_k{k}_query")

elif graph_builder == "shared_attribute":
    sc = cfg["graph"]["shared_attribute"]
    e = SA.build_shared_attribute(bundle.attrs, sc["columns"], allowed, bundle.n, sc["max_clique"], sc["max_degree"], sc["min_group"], seed=0)
    e.save(gdir / "edges_shared_attribute.npz"); built["shared_attribute"] = e
    q = SA.query_shared_attribute(bundle.attrs, sc["columns"], allowed, te, bundle.n, sc["max_degree"], seed=0)
    q.save(gdir / "edges_shared_attribute_query.npz"); built["shared_attribute_query"] = q; full_graphs.add("shared_attribute_query")
    print("group stats:", json.dumps(e.meta.get("stats", {}), indent=1))

elif graph_builder == "relational":
    from src.data import home_credit as HC

    tables = HC.load_relational_tables(bundle.ids, cfg["graph"]["relational"]["max_children_per_customer"])
    print({k: v.shape for k, v in tables.items()})
    full = REL.build_hetero(X, tables, allowed_customers=None, fit_rows=tr)
    REL.save_hetero(full, gdir / "hetero_relational.npz"); stats["relational_full"] = REL.hetero_stats(full)
    clean = REL.build_hetero(X, tables, allowed_customers=allowed, fit_rows=tr)
    REL.save_hetero(clean, gdir / "hetero_relational_clean.npz"); stats["relational_clean"] = REL.hetero_stats(clean)
    for _, rel, _ in REL.RELATIONS:
        assert not np.isin(clean[f"ei_customer__{rel}"][0], te).any(), f"LEAKAGE: test customer in clean {rel}"
    print("clean hetero graph: no test customers OK")
else:
    raise ValueError(f"unknown graph_builder {graph_builder}")

print(f"built in {time.time() - t:.0f}s:", {k: v.n_edges for k, v in built.items()})

## Leakage assertions

Every *clean* graph must have zero edges touching a test position. Query graphs and the real DGraph-Fin graph contain test nodes by design and are labelled as such.

In [ ]:
leak_report = {}
for name, e in built.items():
    if name in full_graphs:
        is_test = np.zeros(e.n_nodes, bool); is_test[te] = True
        leak_report[name] = {"kind": "full/query (test nodes attached for scoring)", "edges_touching_test": int((is_test[e.src] | is_test[e.dst]).sum())}
        # query edges: test rows may appear as sources only, never as both endpoints
        if name.endswith("_query"):
            assert not (is_test[e.src] & is_test[e.dst]).any(), f"LEAKAGE: test-test edge in {name}"
        continue
    assert_no_test_nodes(e, te, name)
    leak_report[name] = {"kind": "clean", "edges_touching_test": 0}
print("all clean graphs pass")
pd.DataFrame(leak_report).T

## Graph statistics

Degree distribution, isolated share, and label homophily measured on **training** labels only: P(neighbour is bad | node is bad) versus the base rate. If that ratio is near 1 the graph carries no label signal and no GNN will find any.

In [ ]:
for name, e in built.items():
    stats[name] = {**degree_stats(e), **label_homophily(e.symmetrized(), y, train_mask)}
    stats[name]["homophily_ratio"] = stats[name].get("p_bad_neighbor_given_bad", np.nan) / max(stats[name].get("base_rate", np.nan), 1e-9)
stats_df = pd.DataFrame(stats).T
stats_df

In [ ]:
homo = {k: v for k, v in built.items() if not k.endswith("_query")}
if homo:
    fig = viz.plot_degree_hist({k: v.degrees() for k, v in homo.items()}, title=dataset)
    viz.savefig(fig, run_dir, "degree_hist")
    plt.show()
    hr = stats_df.loc[list(homo), "homophily_ratio"].astype(float).dropna()
    if len(hr):
        fig, ax = plt.subplots(figsize=(6, 3))
        hr.plot.bar(ax=ax); ax.axhline(1, color="k", lw=0.8)
        ax.set(title="P(bad neighbour | bad) / base rate  (train labels)", ylabel="ratio")
        viz.savefig(fig, run_dir, "homophily")
        plt.show()

In [ ]:
summary = {"graphs": {k: int(v.n_edges) for k, v in built.items()}, "elapsed_s": round(time.time() - t0, 1)}
results.save_metrics(run_dir, {"models": {}, "graph_stats": stats, "leakage": leak_report, "params": {"graph_builder": graph_builder, "quick": quick}})
ckpt.mark_done(run_dir, summary)
summary